# **Feature Engineering**

This notebook will focus on feature engineering.

Deciding on target:

We will pick future_vol_10 as our target.

In [73]:
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import numpy as np

In [25]:
prices = pd.read_csv("C:/Users/jungn/OneDrive/Documents/Projects/Market-Volatility-Prediction/data/spy_eda.csv", index_col=0, parse_dates=True)

C:\Users\jungn\AppData\Local\Temp\ipykernel_24920\3325194594.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  prices = pd.read_csv("C:/Users/jungn/OneDrive/Documents/Projects/Market-Volatility-Prediction/data/spy_eda.csv", index_col=0, parse_dates=True)


In [26]:
prices

,Open,High,Low,Close,Volume,log_return,future_vol_5,future_vol_10,future_vol_20
Price,,,,,,,,,
Ticker,SPY,SPY,SPY,SPY,SPY,NaN,NaN,NaN,NaN
Date,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-02,170.47258874432092,170.88559529706598,168.65534982907482,169.6878662109375,121465900,-0.000535,0.015351,0.011903,0.011045
2015-01-05,168.6470660413276,168.8122661140158,166.3177160273211,166.6233367919922,169632600,-0.018225,0.012987,0.010447,0.010748
2015-01-06,166.9289650053605,167.44935723130908,164.26094653071848,165.05392456054688,209151400,-0.009464,0.011958,0.009956,0.010537
...,...,...,...,...,...,...,...,...,...
2025-12-24,684.313382495532,687.1781631452748,684.164151137988,686.7305297851562,39445600,0.003512,0.003566,0.004751,0.006696
2025-12-26,686.989161229118,688.0037280321676,685.6264081645946,686.660888671875,41613300,-0.000101,0.005342,0.004756,0.006749
2025-12-29,683.9055083361335,685.5567672531378,682.4433081832867,684.2138671875,62559500,-0.003570,0.005762,0.004623,0.006687


In [27]:
prices10 = prices.drop(["future_vol_5", "future_vol_20"], axis = 1) #drop columns we don't need

Now we will add some more features to better predict volatility:

**Return based features**

We will add rolling returns, which tell the model: "Has the market recently moved a lot?"

In [44]:
prices10["return_5d"] = prices10["log_return"].rolling(5).sum()
prices10["return_10d"] = prices10["log_return"].rolling(10).sum()
prices10["return_20d"] = prices10["log_return"].rolling(20).sum()

**Volatility Features**

"How volatile has the market been recently?"

In [45]:
prices10["vol_5"] = prices10["log_return"].rolling(5).std()
prices10["vol_10"] = prices10["log_return"].rolling(10).std()
prices10["vol_20"] = prices10["log_return"].rolling(20).std()

**Price range features**

Large intraday ranges often precede higher volatility.

In [46]:
cols = ["Open", "High", "Low", "Close", "Volume"]

prices10[cols] = prices10[cols].apply(
    pd.to_numeric,
    errors="coerce"
)

In [47]:
prices10["hl_range"] = (
    prices10["High"] - prices10["Low"]
) / prices10["Close"]

In [48]:
prices10["oc_range"] = (prices10["Close"] - prices10["Open"]) / prices10["Open"]

Measures the strength of the daily move.

**Volume Features**

Volume often spikes before volatility.

In [49]:
prices10["vol_change"] = prices10["Volume"].pct_change() #Volume change

In [50]:
prices10["vol_ratio"] = (prices10["Volume"] /prices10["Volume"].rolling(20).mean()) #relative volume ie "is today's volume unusually high?"

In [51]:
prices10.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2767 entries, Date to 2025-12-31
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Open           2766 non-null   float64
 1   High           2766 non-null   float64
 2   Low            2766 non-null   float64
 3   Close          2766 non-null   float64
 4   Volume         2766 non-null   float64
 5   log_return     2766 non-null   float64
 6   future_vol_10  2766 non-null   float64
 7   return_5d      2762 non-null   float64
 8   return_10d     2757 non-null   float64
 9   return_20d     2747 non-null   float64
 10  vol_5          2762 non-null   float64
 11  vol_10         2757 non-null   float64
 12  vol_20         2747 non-null   float64
 13  hl_range       2766 non-null   float64
 14  oc_range       2766 non-null   float64
 15  vol_change     2765 non-null   float64
 16  vol_ratio      2747 non-null   float64
dtypes: float64(17)
memory usage: 389.1+ KB


In [54]:
prices10.isna().sum()

Open              1
High              1
Low               1
Close             1
Volume            1
log_return        1
future_vol_10     1
return_5d         5
return_10d       10
return_20d       20
vol_5             5
vol_10           10
vol_20           20
hl_range          1
oc_range          1
vol_change        2
vol_ratio        20
dtype: int64

In [65]:
prices10 = prices10.dropna()

### **Creating the test set**

For the test set, we will use data from 2026-01-01 to 2026-06-01

In [85]:
test_set = yf.download("SPY", start = "2025-12-24", end = "2026-06-30")

[*********************100%***********************]  1 of 1 completed


In [86]:
test_set["log_return"] = np.log(test_set["Close"] / test_set["Close"].shift(1))

In [87]:
test_set["future_vol_10"] = (test_set["log_return"].rolling(10).std().shift(-10))

In [89]:
test_set = test_set[
    (test_set.index >= "2026-01-01") & (test_set.index <= "2026-06-01")
]

In [90]:
test_set

Price,Close,High,Low,Open,Volume,log_return,future_vol_10
Ticker,SPY,SPY,SPY,SPY,SPY,,
Date,,,,,,,
2026-01-02,679.558594,683.239047,676.226327,682.085206,89377200,0.001831,0.004173
2026-01-05,684.084595,685.785577,682.751712,682.910840,71927200,0.006638,0.007665
2026-01-06,688.152954,688.660268,684.144289,684.293460,69273800,0.005930,0.008424
2026-01-07,685.934753,690.291605,685.676118,688.530942,75588300,-0.003229,0.008590
2026-01-08,685.865112,686.969230,683.855771,685.178757,64019200,-0.000102,0.008591
...,...,...,...,...,...,...,...
2026-05-26,748.661316,750.197337,746.446989,748.082789,41123600,0.006617,0.009273
2026-05-27,748.531616,749.449235,746.297321,748.950520,42106300,-0.000173,0.010244
